In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("liteLLM").setLevel(logging.ERROR)

from litellm import completion


In [2]:
import litellm
litellm.suppress_debug_info=True


In [3]:
import warnings
import logging

# keep the recording clean
warnings.filterwarnings("ignore")
logging.getLogger("liteLLM").setLevel(logging.ERROR)


In [4]:
import os 
from dotenv import load_dotenv
load_dotenv()

print("OpenAI key laoded : True "if os.getenv("OPENAI_API_KEY") else False)
print("GEMINI key laoded : True "if os.getenv("GEMINI_API_KEY") else False )
print("GROQ key laoded : True "if os.getenv("GROQ_API_KEY") else False )

OpenAI key laoded : True 
GEMINI key laoded : True 
GROQ key laoded : True 


In [11]:
from litellm import completion

response_gemini=completion(
        model="gemini/gemini-3.5-flash",

    messages=[{"role":"user","content":"What is RAG in one sentence?"}]
)
print("Gemini Response : ",response_gemini.choices[0].message.content)

response_groq=completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role":"user","content":"What is RAG in one sentence?"}])

print("GROQ Response : ",response_groq.choices[0].message.content)

Gemini Response :  **Retrieval-Augmented Generation (RAG)** is an AI technique that improves the accuracy and reliability of large language models by fetching relevant, up-to-date information from an external knowledge base to ground its generated responses.
GROQ Response :  RAG (Red, Amber, Green) is a widely used status indicator system where Red typically denotes a critical or high-risk issue, Amber (or Yellow) indicates a cautionary or moderate-risk situation, and Green signifies a normal, low-risk, or satisfactory condition.


In [13]:
from litellm import completion

prompt="What is litellm in one sentence?"

providers=[
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/llama-3.3-70b-versatile"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-3.5-flash"),
]

# one loop one function call for multiple providers

for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")
    

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : Littlem is a machine learning library developed by Microsoft, designed to provid
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : ❌ ServiceUnavailableError


In [ ]:
## fall back implementation for multiple providers
## if one provider fails, it will try the next one in the list

from litellm import completion

response=completion(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"What is fallback in llm gateway ?"}],
    fallbacks=[
        "qwen/qwen3-32b",
        "groq/llama-3.3-70b-versatile"
    ]
)
print("Response:",response.choices[0].message.content[:200])
print("Model used:",response.model)

16:01:40 - LiteLLM:ERROR: fallback_utils.py:69 - Fallback attempt failed for model gpt-4o-mini: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Traceback (most recent call last):
  File "c:\Users\vikas\OneDrive\Desktop\Ai Agentic Code\LLM_GateWays\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 930, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\vikas\OneDrive\Desktop\Ai Agentic Code\LLM_GateWays\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vikas\OneDrive\Desktop\Ai Ag

Response: In the context of an LLM (Large Language Model) gateway, a fallback refers to a backup or alternative response mechanism that is triggered when the primary LLM model is unable to generate a response o
Model used: llama-3.3-70b-versatile


In [9]:
from litellm import completion, completion_cost

response = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "What is price tracking in llm gateway?"}
    ]
)

# Pass the explicit model identifier along with the response object
cost = completion_cost(completion_response=response, model="groq/llama-3.3-70b-versatile")


print("Response:", response.choices[0].message.content[:200])
print("Input tokens:", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost: ${cost:.8f} USD")

Response: Price tracking in an LLM (Large Language Model) gateway refers to the process of monitoring and analyzing the prices of various goods, services, or assets in real-time. The LLM gateway is a platform t
Input tokens: 44
Output tokens: 486
Cost: $0.00040990 USD


In [10]:
## caching -> enable in memeory caching with one line
import litellm 

# reset any callbacks left over from earlier calls
litellm.callback=[]
litellm.success_callback=[]
litellm.failure_callback=[]
litellm._async_success_callback=[]
litellm._async_failure_callback=[]

# clearing any router strategy state 
litellm.cache=None
print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [13]:
import litellm 
import time
from litellm import completion
from litellm.caching import Cache

litellm.cache=Cache(type="local")

prompt="What is caching in llm gateway ?"

start=time.time()
r1=completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)

t1=time.time()-start
print(f"❄️  First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

start = time.time()
r2 = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")


❄️  First call (API):   2.15s — In the context of a Large Language Model (LLM) gateway, caching refers to the process of storing frequently accessed data or results in a temporary storage location, called a cache, to improve performance and reduce the latency of subsequent requests.

Here's how caching works in an LLM gateway:

**Why caching is needed:**

1. **Computational intensity**: LLMs are computationally intensive, requiring significant processing power to generate responses to user requests.
2. **Network latency**: User requests need to be sent to the LLM, which can introduce network latency, making the overall response time slower.
3. **High volume of requests**: LLM gateways may receive a large number of requests, which can lead to congestion and increased processing times.

**How caching helps:**

1. **Faster response times**: By storing frequently accessed data in a cache, the LLM gateway can retrieve the results quickly from the cache instead of recomputing them.
2. **Redu

In [ ]:
## smart routing -> differeent models for different use cases i.e for coding -> coding model 
# for fast reply -> lllama


In [19]:
import os 
from litellm import Router

model_list=[
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    
    {
        "model_name": "smart-coding",                                                  # 👈 alias kept
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",                                      # 👈 mapped to llama instead
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)
code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (Gemini):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  **AI is Changing Software – Key Takeaways**

| Aspect | What’s Happening | Why It Matters |
|--------|------------------|----------------|
| **Code Ge

🧠 Smart/coding (Gemini):
 **Reversing a String in Python**

Here's a simple function that takes a string as input and returns the reversed string.

```python
def reverse_string(input_str: str) -> str:
    """
    Reverses the input string.

    Args:
        input_str (str): The string to


In [21]:
## laod balancing 
from litellm import Router
import os 

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        }
    },

    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gemini/gemini-2.5-flash",
            "api_key": os.getenv("GOOGLE_API_KEY"),
        }
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)


print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        b036de5054fc781996bec68c2731616a4e597b5941ee5a65173883516a1971eb   355 ms   Hello, I'm here to assist you with 
#2        b036de5054fc781996bec68c2731616a4e597b5941ee5a65173883516a1971eb   341 ms   Hello. This is request 2. How can I
#3        ac227ef4d5ed557613ae7cb6111e4dc2a71f48a61fb10815dfb6917988de26f4  1122 ms   Hello, request 3
#4        ac227ef4d5ed557613ae7cb6111e4dc2a71f48a61fb10815dfb6917988de26f4  2555 ms   Hello!

I'm ready for "request 4," 
#5        b036de5054fc781996bec68c2731616a4e597b5941ee5a65173883516a1971eb   489 ms   Hello! You've requested 5, but I'm 
#6        b036de5054fc781996bec68c2731616a4e597b5941ee5a65173883516a1971eb   367 ms   Hello. You've requested 6, but I'm 


In [25]:
#  Strategy 1: least-busy —
# The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-2.5-flash",
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 Gemini 2.5 flash"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   # 👈 the magic
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🔵 Gemini 2.5 flash
Request 2 → 🟢 Groq
Request 3 → 🔵 Gemini 2.5 flash
Request 4 → 🟢 Groq
Request 5 → 🔵 Gemini 2.5 flash
Request 6 → 🟢 Groq
Request 7 → 🔵 Gemini 2.5 flash
Request 8 → 🟢 Groq

🎯 Distribution:
   🔵 Gemini 2.5 flash: ████ (4)
   🟢 Groq: ████ (4)


In [26]:
# Strategy 2: latency-based-routing —
# The "Always Pick the Fastest" Pattern The idea: The router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins.
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-2.5-flash",
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 Gemini 2.5 flash"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama-3.3"}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   # 👈 picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🔵 Gemini 2.5 flash                1001 ms
#2    🟢 Groq Llama-3.3                   226 ms
#3    🟢 Groq Llama-3.3                   111 ms
#4    🔵 Gemini 2.5 flash                 999 ms
#5    🟢 Groq Llama-3.3                   112 ms
#6    🟢 Groq Llama-3.3                   324 ms
#7    🟢 Groq Llama-3.3                   139 ms
#8    🟢 Groq Llama-3.3                   217 ms
#9    🟢 Groq Llama-3.3                   222 ms
#10   🟢 Groq Llama-3.3                   216 ms


In [27]:
# 🎯 Strategy 4: cost-based-routing — The "Always Cheapest" Pattern
# The idea: Pick the deployment that costs the least per token right now. Beautiful for cost-sensitive apps.

import os
from litellm import Router

# Different providers with very different price points
model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o",             # ~$2.50/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o (premium)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-2.5-flash",        # ~$0.15/M input tokens
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 Gemini 2.5 flash (cheap)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",   # ~$0.05/M
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama (cheapest)"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"   # 👈 valid strategy
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🟢 Groq Llama (cheapest)
Request 2 → 🔵 Gemini 2.5 flash (cheap)
Request 3 → 🟢 Groq Llama (cheapest)
Request 4 → 🟢 Groq Llama (cheapest)
Request 5 → 🟢 Groq Llama (cheapest)


In [36]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


primary=ChatLiteLLM(model="gpt-4o")

# fall back to gemini if gpt fails
fallback_1=ChatLiteLLM(model="gemini/gemini-2.5-flash")
fallback_2=ChatLiteLLM(model="groq/llama-3.3-70b-versatile")

robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt=ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)


❌ Call failed: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
❌ Call failed: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 8.798084347s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gem

In [37]:
import time
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()

def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error

def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    ["gpt-4o","gpt-4o-mini",   "groq/llama-3.3-70b-versatile"],
        "summary": ["gemini/gemini-2.5-flash","groq/llama-3.3-70b-versatile"],
        "general": ["groq/llama-3.3-70b-versatile", "gpt-4o-mini"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }


# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")

❓ Q: Write a Python function to compute Fibonacci numbers.
❌ Call failed: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
   ⚠️  gpt-4o failed (RateLimitError), trying next...
❌ Call failed: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
   ⚠️  gpt-4o-mini failed (RateLimitError), trying next...
🏷️  Task:    code
🤖 Model:    llama-3.3-70b-versatile
⏱️  Latency: 7.22s
💰 Cost:    n/a
💬 Answer:  **Fibonacci Function in Python**

The following Python function uses recursion to compute Fibonacci numbers. The Fibonacci sequence is a series of numbers where a ...
❓ Q: Summarize the importance of